# Apprentissage non-supervisé : partitionnement par algorithmes des k-moyennes

# Table of contents
1. [Modèle de mélange gaussien (GMM)](#part1)
1. [Algorithme des k-moyennes](#part2)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# %matplotlib inline
sns.set()

from matplotlib import cm
from scipy.stats import multivariate_normal

# Modèle de mélange gaussien (GMM) <a id="part1"></a>


<div class="alert alert-block alert-info">

À l'aide des fonctions suivantes, générer un échantillon de GMM de taille 200 avec les paramètres suivants :
$$
    \begin{cases}
        \pi_1 &= 0.33\\
        \mu_1 &= (0, 0),
    \end{cases}
$$
$$
    \begin{cases}
        \pi_2 &= 0.33\\
        \mu_2 &= (5, 0),
    \end{cases}
$$
$$
    \begin{cases}
        \pi_3 &= 0.34\\
        \mu_3 &= (2, -5),
    \end{cases}
$$
et une matrice de covariance identité.
Afficher les données et les "contours" des matrices de covariance.

<!-- <br> -->
</div>

In [ ]:
def covariance(sigma1=1., sigma2=1., theta=0.):
    """
        Covariance matrix with eigenvalues sigma1 and sigma2, rotated by the angle theta.
    """
    rotation = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta), np.cos(theta)]])
    cov = np.array([[sigma1, 0.],
                   [0, sigma2]])
    return rotation.dot(cov.dot(rotation.T))

def sample_gm(weights, means, covariances, size=1):
    """Sample points from a Gaussian mixture model specified by the weights, the means
    and the covariances. These three parameters are lists."""
    X = None
    p = np.random.multinomial(1, weights, size=size)
    for (m, c, i) in zip(means, covariances, p.T):
        Y = np.random.multivariate_normal(m, c, size=size)
        if X is None:
            X = Y.copy()
        else:
            X[i==1] = Y[i==1]
    return X

def plot_cov(cov, mean=[0, 0], cst=6, num=200, color='r'):
    """Display the ellipse associated to the covariance matrix cov.
    If mean is specified, the ellipse is translated accordingly.
    """
    cov = np.linalg.inv(np.asarray(cov))
    mean = np.asarray(mean)
    theta = np.linspace(0, 2*np.pi, num=num)
    X = np.c_[np.cos(theta), np.sin(theta)]
    X = X.T * np.sqrt(cst / np.diag(X.dot(cov.dot(X.T))))
    X = X.T + mean
    plt.plot(mean[0], mean[1], color+'*', markersize=10)
    plt.plot(X[:, 0], X[:, 1], color)

In [ ]:
# Answer
weights = [0.33, 0.33, 0.34]
means = [[0, 0], [5, 0], [2, -5]]
cov_param = [(1, 1, 0), (1, 1, 0), (1, 1, 0)]  # Parameters for the covariance function

cov_mat = [covariance(*c) for c in cov_param]  # Build the covariance matrices

# To do
X = sample_gm(
    weights=weights,
    means=means,
    covariances=cov_mat, 
    size=200
)
plt.scatter(X[:, 0], X[:, 1])
for cov, mean in zip(cov_mat, means):
    plot_cov(cov, mean)

# End to do
plt.axis('image');

<div class="alert alert-block alert-info">

Estimer les paramètres du modèle avec la classe <a href="http://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html#sklearn.mixture.GaussianMixture">Gaussian mixture</a>, en considérant un mélange à 3 composantes.
    Afficher les probabilités a priori estimées et la valeur finale du critère maximisé.
    Afficher le résultat de l'estimation sur un graphique similaire au précédent.

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=len(means))
gmm.fit(X)

In [ ]:
# probabilités a priori estimées
predicted_probas = gmm.weights_
predicted_probas

In [ ]:
# la valeur finale du critère maximisé. 
gmm.score(X)

In [ ]:
#  Afficher le résultat de l'estimation sur un graphique similaire au précédent.
groups = gmm.predict(X)
plt.scatter(X[:, 0], X[:, 1], c=groups);

plt.scatter(X[:, 0], X[:, 1])
for cov, mean in zip(cov_mat, means):
    plot_cov(cov, mean)

for cov, mean in zip(gmm.covariances_, gmm.means_):
    plot_cov(cov, mean, color="g")

<div class="alert alert-block alert-info">

Répéter cet estimation et évaluer la stabilibilité des résultats.

<!-- <br> -->
</div>

In [ ]:
# Answer
gmm.fit(X)

new_groups = gmm.predict(X)
plt.scatter(X[:, 0], X[:, 1], c=new_groups);

<div class="alert alert-block alert-info">

Que dire de la stabilité de l'estimation si les paramètres initiaux sont choisis au hasard (chercher le paramètre adéquat de <a href="http://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html#sklearn.mixture.GaussianMixture">Gaussian mixture</a>) ?

<!-- <br> -->
</div>

In [ ]:
# Answer

<div class="alert alert-block alert-info">

Compléter le script suivant afin de : <br>
1. échantillonner suivant un GMM ; <br>
2. estimer les paramètres avec <a href="http://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html#sklearn.mixture.GaussianMixture">Gaussian mixture</a> ; <br>
3. afficher le graphique des données et des estimations. <br>

Analyser les résultats et constater la présence d'événements innatendus.
    
<!-- <br> -->
</div>

In [ ]:
# Answer
gmm = GaussianMixture(n_components=2)

for it in range(6):
    plt.figure(figsize=(10, 3))
    for it, (weights, means, covariances) in enumerate([
        ([0.5, 0.5], [[0, 0], [5, 0]], [(1, 1, 0), (1, 1, 0)]),
        ([0.05, 0.95], [[0, 0], [5, 0]], [(1, 1, 0), (1, 1, 0)]),
        ([0.5, 0.5], [[0, 0], [0, 0]], [(10, 1, 0), (1, 10, 0)]),
        ([0.5, 0.5], [[0, 0], [5, -5]], [(10, 1, 0), (1, 10, 0)])]):
        # To do

        # End to do
        plt.subplot(1, 4, it+1)
        plt.scatter(X[:, 0], X[:, 1])
        # To do

        # End to do 

# Algorithme des k-moyennes <a id="part2"></a>


<div class="alert alert-block alert-info">

À partir des données suivantes, estimer les paramètres d'un GMM.
    Afficher le résultat de l'estimation et ajouter le paritionnement qui en découle avec la fonction suivante.

<!-- <br> -->
</div>

In [ ]:
def map_regions(clf, data=None, num=500):
    """
        Map the regions f(x)=1…K of the classifier clf within the same range as the one
        of the data.
        Input:
            clf: classifier with a method predict
            data: input data (X)
            num: discretization parameter
    """
    xmin, ymin = data.min(axis=0)
    xmax, ymax = data.max(axis=0)
    x, y = np.meshgrid(np.linspace(xmin, xmax, num), np.linspace(ymin, ymax))
    z = clf.predict(np.c_[x.ravel(), y.ravel()]).reshape(x.shape)
    zmin, zmax = z.min(), z.max()
    plt.imshow(z, origin='lower', interpolation="nearest",
               extent=[xmin, xmax, ymin, ymax], cmap=cm.coolwarm,
              alpha=0.3)

In [ ]:
(weights, means, covariances) = ([0.3, 0.2, 0.5], [[-5, -1], [5, 0], [2, -5]],
                                 [(1, 5, np.pi/3), (1, 5, np.pi/3), (5, 1, np.pi/3)])
X = sample_gm(weights, means, [covariance(*c) for c in covariances], size=200)

In [ ]:
# Answer

<div class="alert alert-block alert-info">

Faire de même avec <a href="http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html">k-means</a>.
    Quelles différences constatez-vous ?

<!-- <br> -->
</div>

In [ ]:
# Answer

<div class="alert alert-block alert-info">

À partir du jeu de données suivant, lancer plusieurs fois l'algorithme de partionnement <a href="http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html">k-means</a> avec une initialisation aléatoire (chercher le paramètre adéquat) et afficher les résultats.
    Qu'observez-vous ?

<!-- <br> -->
</div>

In [ ]:
(weights, means, covariances) = ([0.05, 0.2, 0.75], [[-5, -1], [5, 0], [2, -5]],
                                 [(1, 5, np.pi/3), (1, 5, np.pi/3), (5, 1, np.pi/3)])
X = sample_gm(weights, means, [covariance(*c) for c in covariances], size=100)

In [ ]:
# Answer

<div class="alert alert-block alert-info">

On cherche ici à analyser le comportement de <a href="http://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html#sklearn.mixture.GaussianMixture">Gaussian mixture</a> et de <a href="http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html">k-means</a> pour les groupes non-convexes.
Pour ce faire : <br>    
1. générer des <a href="http://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html">lunes</a> (puis des <a href="http://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_circles.html#sklearn.datasets.make_circles">cercles</a>) avec un bruit réglé à $0.1$ ; <br>
2. afficher les deux groupes de points avec `plotXY` ; <br>
3. afficher le partitionnement retourné par <a href="http://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html#sklearn.mixture.GaussianMixture">Gaussian mixture</a> puis par <a href="http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html">k-means</a>.  <br>

Qu'observez-vous ?

<!-- <br> -->
</div>

In [ ]:
def plotXY(X, Y, legend=True):
    """
        Scatter points with a color for each class.
        Input:
            X and Y may be:
            - two numpy arrays with two columns; each array is the data matrix for a class (works only for
            two classes).
            - a numpy array with two columns (the data matrix) and the vector of labels (works for many classes).
    """    
    if Y.ndim > 1:
        X1 = X
        X2 = Y
        XX = np.concatenate((X, Y), axis=0)
        YY = np.concatenate((np.ones(X.shape[0]), -np.ones(Y.shape[0])))
    else:
        XX = X
        YY = Y
    for icl, cl in enumerate(np.unique(YY)):
        plt.scatter(XX[YY==cl, 0], XX[YY==cl, 1], label='Class {0:d}'.format(icl+1))
    plt.axis('equal')
    if legend:
        plt.legend()

In [ ]:
# Answer